# 世界通史时空图谱 · 零延迟中文纪录片语音预生成工厂（Colab）

为图谱全部 **257 项核心词条**（170 项重大史事 + 87 位历史人物，每项含 3 幕分段解说，共 **771 条音频片段**）批量预生成高质感中文纪录片语音并压制为轻量 MP3（48kbps 单声道，全部仅约 **25~35 MB**）。

导入项目后，点击「AI 语音解说」或播放「电影演示」时会**零延迟秒播预生成音频，不消耗 Gemini 实时 TTS 额度**！

### 双引擎可选
- **`edge-tts`（默认推荐 · 极速）**：微软神经纪录片解说音色（如 `zh-CN-YunjianNeural` 沉稳男声 / `zh-CN-XiaoxiaoNeural` 温润女声），全量 771 条仅需约 **3 分钟**。
- **`f5-tts`（Colab GPU 声音克隆可选）**：在 L4/T4 GPU 上用自备 8 秒中文纪录片旁白参考音频克隆专属声线。

### 回到本地一键导入
```bash
npm run media:import -- ~/Downloads/atlas-audio-pack.zip
```

In [ ]:
#@title 1. 挂载 Google Drive 缓存目录并检查环境
import os, shutil, torch

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_SUBDIR = "world-history-atlas/audio"  #@param {type:"string"}

if torch.cuda.is_available():
    print(f"✅ GPU 就绪: {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️ 当前为 CPU 运行时（使用 edge-tts 引擎完全足够；若启用 f5-tts 声音克隆请切换 GPU）。")

WORK_DIR = "/content/atlas-audio"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = os.path.join("/content/drive/MyDrive", DRIVE_SUBDIR)
        print(f"📁 已绑定断点缓存目录: {WORK_DIR}")
    except Exception as exc:
        print(f"Drive 挂载跳过，使用临时目录 {WORK_DIR}: {exc}")

os.makedirs(WORK_DIR, exist_ok=True)
print("ffmpeg:", shutil.which("ffmpeg") or "未找到（将在下一步安装）")

In [ ]:
#@title 2. 选择语音引擎并安装依赖
TTS_ENGINE = "edge-tts"  #@param ["edge-tts", "f5-tts"]
VOICE_PRESET = "zh-CN-YunjianNeural"  #@param ["zh-CN-YunjianNeural", "zh-CN-YunxiNeural", "zh-CN-XiaoxiaoNeural", "zh-CN-XiaoyiNeural"]
SPEECH_RATE = "+4%"  #@param {type:"string"}
MP3_BITRATE = "48k"  #@param ["32k", "48k", "64k"]

!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install --upgrade edge-tts mutagen nest_asyncio
if TTS_ENGINE == "f5-tts":
    !pip -q install --upgrade f5-tts
print(f"✅ 依赖安装完成 | 当前引擎: {TTS_ENGINE} | 音色: {VOICE_PRESET}")

In [ ]:
#@title 3. 拉取纪录片旁白脚本清单 (narration-scripts.json)
import json, urllib.request

RAW_SCRIPTS_URL = "https://raw.githubusercontent.com/quanliulq-pixel/world-history-atlas/main/colab/narration-scripts.json"  #@param {type:"string"}
LOCAL_SCRIPTS_PATH = os.path.join(WORK_DIR, "narration-scripts.json")

try:
    urllib.request.urlretrieve(RAW_SCRIPTS_URL, LOCAL_SCRIPTS_PATH)
    print(f"✅ 已拉取脚本清单: {LOCAL_SCRIPTS_PATH}")
except Exception as err:
    print(f"⚠️ 无法直接联网拉取 ({err})，请上传 colab/narration-scripts.json")
    from google.colab import files
    uploaded = files.upload()
    first_name = next(iter(uploaded.keys()))
    with open(LOCAL_SCRIPTS_PATH, "wb") as f:
        f.write(uploaded[first_name])

with open(LOCAL_SCRIPTS_PATH, "r", encoding="utf-8") as f:
    script_catalog = json.load(f)

print("📊 脚本统计:", script_catalog.get("counts", {}))

In [ ]:
#@title 4. 若选择 f5-tts 声音克隆，请在此上传 5~12 秒参考人声 WAV（edge-tts 可直接跳过）
REF_AUDIO_PATH = ""
REF_TEXT = "公元前两千五百年，大河流域的城邦秩序正悄然重塑人类文明的版图。"  #@param {type:"string"}
f5_model = None

if TTS_ENGINE == "f5-tts":
    from google.colab import files
    from f5_tts.api import F5TTS
    print("请上传参考人声 WAV/MP3 文件（8 秒左右清晰中文男/女旁白最佳）:")
    up = files.upload()
    REF_AUDIO_PATH = list(up.keys())[0]
    f5_model = F5TTS()
    print("✅ F5-TTS 模型与参考声线加载完成:", REF_AUDIO_PATH)
else:
    print("ℹ️ 使用 edge-tts 神经解说音色，无需参考音频。")

In [ ]:
#@title 5. 批量合成音频并实时写入 manifest.json（支持断点续传 + 试听）
import asyncio, datetime, subprocess, time
import nest_asyncio
import edge_tts
from mutagen.mp3 import MP3
from IPython.display import Audio, display

nest_asyncio.apply()

BATCH_MODE = "all"  #@param ["all", "core", "starter"]
OVERWRITE_EXISTING = False  #@param {type:"boolean"}
MAX_ITEMS = 0  #@param {type:"integer"}

all_items = script_catalog["items"]
if BATCH_MODE == "starter":
    selected_items = [x for x in all_items if x.get("tier") == "starter"]
elif BATCH_MODE == "core":
    selected_items = [x for x in all_items if x.get("tier") in ("starter", "core")]
else:
    selected_items = list(all_items)

if MAX_ITEMS and MAX_ITEMS > 0:
    selected_items = selected_items[:MAX_ITEMS]

MANIFEST_PATH = os.path.join(WORK_DIR, "manifest.json")
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        manifest = json.load(f)
else:
    manifest = {"version": 1, "engine": TTS_ENGINE, "items": {}}

manifest["engine"] = f"{TTS_ENGINE}:{VOICE_PRESET if TTS_ENGINE == 'edge-tts' else 'clone'}"
manifest_items = manifest.setdefault("items", {})

def save_manifest():
    manifest["generatedAt"] = datetime.datetime.utcnow().isoformat() + "Z"
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

async def synth_edge_clip(text: str, out_mp3: str):
    tmp_mp3 = out_mp3 + ".raw.mp3"
    comm = edge_tts.Communicate(text, VOICE_PRESET, rate=SPEECH_RATE)
    await comm.save(tmp_mp3)
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", tmp_mp3, "-ac", "1", "-ar", "24000", "-b:a", MP3_BITRATE, out_mp3],
        check=True,
    )
    os.remove(tmp_mp3)

def synth_f5_clip(text: str, out_mp3: str):
    tmp_wav = out_mp3 + ".raw.wav"
    f5_model.infer(ref_file=REF_AUDIO_PATH, ref_text=REF_TEXT, gen_text=text, file_wave=tmp_wav)
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", tmp_wav, "-ac", "1", "-ar", "24000", "-b:a", MP3_BITRATE, out_mp3],
        check=True,
    )
    os.remove(tmp_wav)

def get_duration(mp3_path: str) -> float:
    try:
        return round(float(MP3(mp3_path).info.length), 2)
    except Exception:
        return 0.0

async def run_batch():
    start_ts = time.time()
    new_clips = 0
    skipped_clips = 0
    preview_file = None

    for idx, item in enumerate(selected_items, start=1):
        beat_entries = []
        for beat in item["beats"]:
            rel_file = beat["file"]
            abs_file = os.path.join(WORK_DIR, rel_file)
            os.makedirs(os.path.dirname(abs_file), exist_ok=True)

            if not os.path.exists(abs_file) or OVERWRITE_EXISTING:
                if TTS_ENGINE == "f5-tts" and f5_model is not None:
                    synth_f5_clip(beat["narration"], abs_file)
                else:
                    await synth_edge_clip(beat["narration"], abs_file)
                new_clips += 1
            else:
                skipped_clips += 1

            if preview_file is None:
                preview_file = abs_file

            beat_entries.append({
                "title": beat["title"],
                "narration": beat["narration"],
                "file": rel_file,
                "durationSec": get_duration(abs_file),
            })

        manifest_items[item["key"]] = {
            "kind": item["kind"],
            "id": item["id"],
            "script": item["script"],
            "voice": VOICE_PRESET if TTS_ENGINE == "edge-tts" else "f5-clone",
            "beats": beat_entries,
        }

        if idx % 10 == 0 or idx == len(selected_items):
            save_manifest()
            elapsed = time.time() - start_ts
            print(f"[{idx}/{len(selected_items)}] ✅ {item['title']} | 新合成 {new_clips} 段 | 耗时 {elapsed:.1f}s")

    save_manifest()
    print(f"\n🎉 合成完成！新生成 {new_clips} 段，跳过已有 {skipped_clips} 段，manifest 共 {len(manifest_items)} 个词条。")
    if preview_file:
        print("🔊 试听首段成品:")
        display(Audio(preview_file))

asyncio.get_event_loop().run_until_complete(run_batch())

In [ ]:
#@title 6. 打包导出 atlas-audio-pack.zip 并下载到本地
import zipfile

ZIP_PATH = "/content/atlas-audio-pack.zip"
file_count = 0

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(MANIFEST_PATH, "manifest.json")
    for entry in manifest_items.values():
        for beat in entry.get("beats", []):
            rel = beat["file"]
            abs_path = os.path.join(WORK_DIR, rel)
            if os.path.exists(abs_path):
                zf.write(abs_path, rel)
                file_count += 1

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"📦 已打包 {file_count} 段音频 -> {ZIP_PATH} ({size_mb:.2f} MB)")

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as exc:
    print(f"请手动下载 {ZIP_PATH}: {exc}")